# Colibrí: dataset Roboflow → YOLO26 → tracking → trayectorias

Notebook del backlog **v0.4**. Cada sección indica la historia que cubre.

| Sección | Historia | Requiere video |
|---|---|---|
| 1. Configuración | DoD-03 | no |
| 2. Descarga (ds1 + ds2) | US-01.4 (CA-01.4.1) | no |
| 3. Auditoría del dataset | US-01.4 (CA-01.4.2 a 01.4.7) | no |
| 4. Particiones + prueba de video propia | US-01.5 | frames ya extraídos |
| 5. Línea base YOLO26 COCO `bird` | US-02.2 (CA-02.2.1) | no |
| 6. Entrenamiento YOLO26 | US-02.2 | no |
| 7. Evaluación provisional | US-02.3 | no (decisión final sí) |
| 8. Tracking ByteTrack vs OC-SORT | US-02.1 / US-02.4 / US-02.5 | **sí** |

Reglas: la API key nunca se escribe en el notebook; la trayectoria detallada nunca se sobrescribe; todo lo provisional se marca como tal.

## 1. Configuración e instalación

In [ ]:
%pip install -q "ultralytics>=8.4" roboflow imagehash pyyaml pandas matplotlib

In [ ]:
import os, json, random, shutil, hashlib, platform, getpass, datetime as dt
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)

WORK = Path("colibri_work")          # cambia a una ruta de Drive si usas Colab
DATA_RAW = WORK / "data" / "roboflow_raw"          # una subcarpeta por dataset
DATA_FINAL = WORK / "data" / "final"
RUNS = WORK / "runs"
REPORTS = WORK / "reports"
for p in (DATA_RAW, DATA_FINAL, RUNS, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

# Decisión 2026-11-01: entrenar con dataset 2 (fotos variadas) + dataset 1 (frames de video en vuelo).
# Si haces fork a tu workspace con una versión sin "Stretch", cambia workspace/project/version aquí.
DATASETS = {
    "ds1": dict(workspace="magic-workspace", project="hummingbird-dataset-0zvvn", version=1, format="yolo26",
                tipo="frames de 2 videos", stretched_from_16_9=True),
    "ds2": dict(workspace="modelodevisinartificialcolibres", project="hummingbirds-detect-classify-gcm5s", version=1,
                format="yolo26", tipo="fotos variadas", stretched_from_16_9=False),
}
CLASS_NAMES = {0: "hummingbird"}      # ds1 'hummingbird' y ds2 'colibri' se unifican en la clase 0
COCO_BIRD_ID = 14                     # clase 'bird' en COCO
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
# Auditoría 2026-11-01: la v1 se exporta con "Stretch to 640x640" desde frames 16:9 (1920x1080 y 1280x720).
ORIG_ASPECT = 16 / 9                  # se usa para medir cajas sin deformación y para des-estirar al materializar
UNSTRETCH_SIZE = (640, 360)           # (ancho, alto) al que se devuelven las imágenes; las etiquetas normalizadas no cambian

RUN_RECORD = {"created": dt.datetime.now().isoformat(timespec="seconds"), "seed": SEED,
              "python": platform.python_version(),
              "datasets": {k: dict(v) for k, v in DATASETS.items()}}

## 2. Descarga de los datasets (US-01.4, CA-01.4.1)

La API key se pide en tiempo de ejecución o se lee de la variable de entorno `ROBOFLOW_API_KEY` (en Colab: *Secrets*). No la pegues en una celda.

In [ ]:
import roboflow
from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Roboflow API key: ")
rf = Roboflow(api_key=api_key)
rf_versions, RAW_DIRS = {}, {}
for key, d in DATASETS.items():
    v = rf.workspace(d["workspace"]).project(d["project"]).version(d["version"])
    ds = v.download(d["format"], location=str(DATA_RAW / key), overwrite=False)
    rf_versions[key], RAW_DIRS[key] = v, Path(ds.location)
    RUN_RECORD["datasets"][key].update({"downloaded": dt.datetime.now().isoformat(timespec="seconds"),
                                        "location": ds.location})
del api_key
RUN_RECORD["roboflow_pkg"] = roboflow.__version__
print(RAW_DIRS)


## 3. Auditoría del dataset (US-01.4)

### 3.1 Licencia y metadatos de exportación (CA-01.4.2, CA-01.4.3)
Roboflow incluye `README.roboflow.txt` / `README.dataset.txt` con la licencia y los pasos de preprocesado y aumentos. **Verifica la licencia antes de seguir: es un gate.**

In [ ]:
for key, root in RAW_DIRS.items():
    print(f"########## {key}: {DATASETS[key]['project']}")
    for readme in sorted(root.glob("README*.txt")):
        print(f"===== {readme.name} =====")
        print(readme.read_text(encoding="utf-8", errors="replace"))
    for attr in ("license", "preprocessing", "augmentation", "splits", "images"):
        if hasattr(rf_versions[key], attr):
            print(attr, "->", getattr(rf_versions[key], attr))


In [ ]:
# La API de Roboflow declara "MIT" (declarada por quien subió el dataset). Los frames salen de dos videos de terceros
# ("videoplayback.mp4" y "190921_SonyClip#1_ParcelaI_visitas.mp4"): la licencia no acredita los derechos de esos videos.
# También se pueden pasar como variables de entorno LICENSE_DS1 / LICENSE_DS2 (útil con run_pipeline.py).
# ds2 declara "CC BY 4.0": exige citar el dataset y su autor en la documentación del proyecto.
LICENSE_CONFIRMED = {"ds1": os.environ.get("LICENSE_DS1"),   # p. ej. "MIT (declarada en Roboflow; procedencia de videos sin verificar)"
                     "ds2": os.environ.get("LICENSE_DS2")}   # p. ej. "CC BY 4.0, atribución en README del proyecto"
assert all(LICENSE_CONFIRMED.values()), "Gate CA-01.4.2: registra la licencia verificada de cada dataset."
for k, lic in LICENSE_CONFIRMED.items():
    RUN_RECORD["datasets"][k]["license"] = lic

### 3.2 Inventario: clases, imágenes, instancias, cajas

In [ ]:
def load_data_yaml(root: Path) -> dict:
    cfg = yaml.safe_load((root / "data.yaml").read_text())
    names = cfg["names"]
    cfg["names"] = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    return cfg

def split_dirs(root: Path) -> dict:
    """Devuelve {split: (dir_imagenes, dir_etiquetas)} para la estructura de Roboflow (train/valid/test)."""
    out = {}
    for split in ("train", "valid", "val", "test"):
        img = root / split / "images"
        if img.is_dir():
            out["valid" if split == "val" else split] = (img, root / split / "labels")
    return out

def read_labels(lbl_path: Path) -> list:
    if not lbl_path.exists():
        return []
    rows = []
    for line in lbl_path.read_text().splitlines():
        parts = line.split()
        if len(parts) == 5:
            c, x, y, w, h = parts
            rows.append((int(c), float(x), float(y), float(w), float(h)))
        elif len(parts) > 5:  # polígono: se convierte a caja envolvente
            c, pts = int(parts[0]), np.array(parts[1:], float).reshape(-1, 2)
            (x0, y0), (x1, y1) = pts.min(0), pts.max(0)
            rows.append((c, (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0))
    return rows

def original_stem(img_path: Path) -> str:
    """Roboflow nombra los archivos '<original>_<ext>.rf.<hash>.<ext>': las copias aumentadas comparten el original."""
    return img_path.name.split(".rf.")[0]

import re
FRAME_RE = re.compile(r"^(?P<src>.+?)[-_](?P<frame>\d+)_(?:jpe?g|png)$", re.I)

def video_frame(stem: str):
    """'videoplayback_mp4-0203_jpg' -> ('videoplayback_mp4', 203). Todo el dataset son frames de 2 videos."""
    m = FRAME_RE.match(stem)
    return (m["src"], int(m["frame"])) if m else (stem, -1)

def inventory(root: Path, ds: str):
    img_rows, box_rows = [], []
    for split, (img_dir, lbl_dir) in split_dirs(root).items():
        for img in sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS):
            with Image.open(img) as im:
                W, H = im.size
            labels = read_labels(lbl_dir / f"{img.stem}.txt")
            stem = original_stem(img)
            src, frame = video_frame(stem) if DATASETS[ds]["stretched_from_16_9"] else (f"{ds}_foto", -1)
            img_rows.append(dict(ds=ds, split=split, path=str(img), stem=stem, src=src, frame=frame, W=W, H=H,
                                 n_boxes=len(labels)))
            for c, x, y, w, h in labels:
                box_rows.append(dict(ds=ds, split=split, path=str(img), cls=c, cx=x, cy=y, w=w, h=h,
                                     area=w * h, aspect=(w * W) / max(h * H, 1e-9), w_px=w * W, h_px=h * H,
                                     aspect_orig=((w * ORIG_ASPECT) / max(h, 1e-9)) if (DATASETS[ds]["stretched_from_16_9"] and W == H)
                                     else (w * W) / max(h * H, 1e-9)))   # aspecto real (ds2 estirado: no recuperable)
    return pd.DataFrame(img_rows), pd.DataFrame(box_rows)

parts = {k: inventory(root, k) for k, root in RAW_DIRS.items()}
for k, root in RAW_DIRS.items():
    names = load_data_yaml(root)["names"]
    print(k, "clases:", names)
    assert len(names) == 1, f"{k}: se esperaba una sola clase; revisa el mapeo a CLASS_NAMES"
imgs = pd.concat([p[0] for p in parts.values()], ignore_index=True)
boxes = pd.concat([p[1] for p in parts.values()], ignore_index=True)
summary = imgs.groupby(["ds", "split"]).agg(imagenes=("path", "size"), sin_etiqueta=("n_boxes", lambda s: int((s == 0).sum())),
                                    multi_ave=("n_boxes", lambda s: int((s > 1).sum())), instancias=("n_boxes", "sum"))
summary["pct"] = (100 * summary["imagenes"] / summary.groupby(level=0)["imagenes"].transform("sum")).round(1)
display(summary)
display(imgs.groupby("ds")[["W", "H"]].describe().T)
display(pd.crosstab(imgs.src, imgs.split, margins=True))   # esperado: 2 videos, ambos en los tres splits
display(boxes.groupby(["ds", "split"])[["w_px", "h_px", "area", "aspect", "aspect_orig"]].describe().T.round(3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(np.sqrt(boxes["area"]), bins=40); ax[0].set_title("sqrt(área normalizada) de cajas")
ax[1].hist(boxes["aspect_orig"], bins=40); ax[1].set_title("aspecto w/h en el frame 16:9 original")
ax[2].scatter(boxes["cx"], boxes["cy"], s=3, alpha=.4); ax[2].invert_yaxis(); ax[2].set_title("centros de caja")
plt.tight_layout(); plt.savefig(REPORTS / "dataset_cajas.png", dpi=120); plt.show()
"""Ojo: boxes["aspect"] está medido sobre la imagen estirada a 640x640 y sale 16/9 veces más estrecho; usar aspect_orig.
Lectura sugerida (CA-01.4.5): una distribución de aspecto bimodal suele separar aves posadas
(cajas más altas) de aves en vuelo con alas extendidas (cajas más anchas). Esa variación de caja
es la que desplazará el centro en el video y debe anotarse como limitación de la referencia corporal."""

### 3.3 Revisión visual con semilla fija (CA-01.4.4)

In [ ]:
def show_grid(paths, lbl_lookup, n_cols=6, title="", out=None):
    n = len(paths); rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(rows, n_cols, figsize=(3 * n_cols, 3 * rows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, p in zip(np.atleast_1d(axes).ravel(), paths):
        im = Image.open(p).convert("RGB"); W, H = im.size
        ax.imshow(im)
        for c, x, y, w, h in lbl_lookup(Path(p)):
            ax.add_patch(plt.Rectangle(((x - w / 2) * W, (y - h / 2) * H), w * W, h * H, fill=False, lw=2, ec="lime"))
            ax.plot(x * W, y * H, "r+", ms=10)          # centro de caja = referencia corporal (US-01.2)
        ax.set_title(Path(p).name[:22], fontsize=7)
    fig.suptitle(title); plt.tight_layout()
    if out: plt.savefig(out, dpi=110)
    plt.show()

def raw_label_lookup(img_path: Path):
    return read_labels(img_path.parent.parent / "labels" / f"{img_path.stem}.txt")

for split in imgs["split"].unique():
    sample = imgs[imgs.split == split].sample(min(24, (imgs.split == split).sum()), random_state=SEED)["path"].tolist()
    show_grid(sample, raw_label_lookup, title=f"Muestra {split}", out=REPORTS / f"muestra_{split}.png")

REVIEW_NOTES = {  # completa tras mirar las rejillas
    "todas_las_aves_etiquetadas": None,
    "criterio_caja_consistente": None,
    "cajas_incluyen_alas_extendidas": None,
    "observaciones": "",
}

### 3.4 Fugas y casi-duplicados (CA-01.4.6)

In [ ]:
# Auditoría 2026-11-01: el dataset son frames consecutivos (paso de 1 frame) de 2 videos con cámara fija,
# repartidos al azar entre splits. Cada imagen de valid/test está a 1-4 frames de una de train: fuga casi total.
# Un pHash con unión transitiva encadenaría cada video entero en un solo grupo, así que la fuga se mide
# por vecindad temporal dentro del mismo video.
LEAK_GAP_FRAMES = 15      # dos frames a <= 15 frames (~0.5 s) se consideran casi-duplicados

def min_gap_to(df: pd.DataFrame, split_col: str, target: str = "train") -> pd.Series:
    """Distancia en frames de cada imagen a la imagen más cercana del mismo video asignada a `target`."""
    out = pd.Series(np.inf, index=df.index)
    for src, g in df[df.frame >= 0].groupby("src"):
        ref = np.sort(g.loc[g[split_col] == target, "frame"].values)
        if len(ref) == 0:
            continue
        pos = np.searchsorted(ref, g.frame.values)
        lo = ref[np.clip(pos - 1, 0, len(ref) - 1)]; hi = ref[np.clip(pos, 0, len(ref) - 1)]
        out[g.index] = np.minimum(np.abs(g.frame.values - lo), np.abs(g.frame.values - hi))
    return out

imgs = imgs.reset_index(drop=True)
imgs["gap_to_train"] = min_gap_to(imgs, "split")
# Solo ds1 son frames de video; en ds2 (fotos) la auditoría por embeddings dio 0,8 % de casi-duplicados entre splits.
ev = imgs[(imgs.ds == "ds1") & imgs.split.isin(["valid", "test"])]
leaks = ev[ev.gap_to_train <= LEAK_GAP_FRAMES]
leaky_groups = leaks.index          # se mantiene el nombre que usa la sección 4
print(f"ds1: valid/test a <= {LEAK_GAP_FRAMES} frames de train: {len(leaks)} de {len(ev)} ({len(leaks)/max(len(ev),1):.0%})")
display(ev.groupby("split").gap_to_train.describe())
# Diagnóstico visual: un frame de test junto a sus vecinos de train
if len(leaks):
    r = leaks.iloc[0]
    neigh = imgs[(imgs.src == r.src) & (imgs.split == "train")].assign(d=lambda d: (d.frame - r.frame).abs()).nsmallest(3, "d")
    show_grid([r.path] + neigh.path.tolist(), raw_label_lookup, n_cols=4, title=f"Fuga: {r.split} frame {r.frame} y vecinos de train")


## 4. Particiones (US-01.5)

**ds2** (fotos, sin fugas) conserva su distribución de Roboflow. Lo que sigue aplica a **ds1**.

Regla documentada (CA-01.5.1): se usa la distribución de Roboflow **solo si** no hay fugas y valid/test tienen al menos `MIN_EVAL_IMAGES`.
La auditoría del 2026-11-01 encontró fuga casi total (frames vecinos del mismo video en train y en valid/test), así que se **re-particiona por bloques temporales**:
cada video se corta en tramos contiguos de unos `CHUNK_IMAGES` frames, los tramos se reparten 70/15/15 **dentro de cada video** (los dos videos aparecen en los tres splits)
y se descartan de valid/test los frames a menos de `GUARD_FRAMES` de un tramo de train.

Limitación que no se puede quitar: ambos videos son de cámara fija, así que valid/test comparten fondo con train. La prueba mide
"mismo escenario, otro momento", no generalización a una escena nueva; para eso está el reporte cruzado entre videos y, después, el video del proyecto.


In [ ]:
MIN_EVAL_IMAGES = 60
RESPLIT_RATIOS = {"train": 0.70, "valid": 0.15, "test": 0.15}
SEGMENT_GAP = 15       # un salto > 15 frames abre un tramo nuevo
CHUNK_IMAGES = 40      # tamaño aproximado de bloque dentro de un tramo largo
GUARD_FRAMES = 15      # banda de guarda entre bloques de distinto split

def temporal_blocks(df: pd.DataFrame) -> pd.Series:
    """Id de bloque temporal por imagen: tramos contiguos de cada video, troceados en ~CHUNK_IMAGES."""
    out = pd.Series(index=df.index, dtype=object)
    for src, g in df.sort_values(["src", "frame"]).groupby("src"):
        seg = (g.frame.diff().fillna(0) > SEGMENT_GAP).cumsum()
        k = 0
        for _, gg in g.groupby(seg):
            for part in np.array_split(gg.index.values, max(1, round(len(gg) / CHUNK_IMAGES))):
                out[part] = f"{src}#{k}"; k += 1
    return out

def block_split(df: pd.DataFrame, ratios: dict, seed: int, tries: int = 3000) -> pd.Series:
    """Asigna bloques a splits por video; elige, entre `tries` sorteos con semilla fija, el más cercano a las cuotas."""
    rng, out = np.random.default_rng(seed), pd.Series(index=df.index, dtype=object)
    for src, g in df.groupby("src"):
        sizes = g.groupby("block").size(); best = None
        for _ in range(tries):
            lab = rng.choice(list(ratios), size=len(sizes), p=list(ratios.values()))
            n = {k: sizes[lab == k].sum() for k in ratios}
            if min(n.values()) == 0:
                continue
            err = sum(abs(n[k] / len(g) - ratios[k]) for k in ratios)
            if best is None or err < best[0]:
                best = (err, dict(zip(sizes.index, lab)))
        out[g.index] = g.block.map(best[1])
    return out

v1 = imgs.ds == "ds1"
sizes = imgs[v1].split.value_counts()
use_original = len(leaky_groups) == 0 and all(sizes.get(s, 0) >= MIN_EVAL_IMAGES for s in ("valid", "test"))
imgs["block"] = None; imgs["final_split"] = imgs["split"]          # ds2: distribución original
imgs.loc[v1, "block"] = temporal_blocks(imgs[v1])
if not use_original:
    imgs.loc[v1, "final_split"] = block_split(imgs[v1], RESPLIT_RATIOS, SEED)

# Banda de guarda: fuera de valid/test lo que quede a <= GUARD_FRAMES de train
imgs["gap_final"] = min_gap_to(imgs, "final_split")
guard = v1 & imgs.final_split.isin(["valid", "test"]) & (imgs.gap_final <= GUARD_FRAMES)
imgs.loc[guard, "final_split"] = "dropped_guard"

# CA-01.5.2: en valid/test solo una imagen por original de Roboflow. Ninguna versión trae aumentos; solo se aplica
# a ds1, porque en ds2 los nombres originales se repiten entre fotos distintas (4675 únicos de 6766).
eval_mask = v1 & imgs.final_split.isin(["valid", "test"])
dupe_aug = eval_mask & imgs.duplicated(subset=["stem"], keep="first")
imgs.loc[dupe_aug, "final_split"] = "dropped_aug_eval"

SPLIT_DECISION = {"use_original_split": bool(use_original), "leaky_eval_images": int(len(leaky_groups)),
                  "method": None if use_original else "bloques temporales por video + banda de guarda",
                  "segment_gap": SEGMENT_GAP, "chunk_images": CHUNK_IMAGES, "guard_frames": GUARD_FRAMES,
                  "min_eval_images": MIN_EVAL_IMAGES, "ratios": None if use_original else RESPLIT_RATIOS,
                  "dropped_guard": int(guard.sum()), "dropped_augmented_from_eval": int(dupe_aug.sum())}
print(SPLIT_DECISION)
display(pd.crosstab([imgs.ds, imgs.src, imgs.split], imgs.final_split, margins=True))
assert (imgs.loc[v1 & imgs.final_split.isin(["valid", "test"]), "gap_final"] > GUARD_FRAMES).all()
for s in ("valid", "test"):
    if (v1 & (imgs.final_split == s)).sum() < MIN_EVAL_IMAGES:
        print(f"Aviso: {s} tiene menos de {MIN_EVAL_IMAGES} imágenes; ajusta CHUNK_IMAGES o GUARD_FRAMES y documenta.")


In [ ]:
def materialize(df: pd.DataFrame, dst: Path, names: dict) -> Path:
    if dst.exists():
        shutil.rmtree(dst)
    manifest = []
    for _, r in df[df.final_split.isin(["train", "valid", "test"])].iterrows():
        src = Path(r.path); lbl = src.parent.parent / "labels" / f"{src.stem}.txt"
        out_name = f"{r.ds}__{src.name}"                       # prefijo: evita choques de nombre entre datasets
        (dst / r.final_split / "images").mkdir(parents=True, exist_ok=True)
        (dst / r.final_split / "labels").mkdir(parents=True, exist_ok=True)
        out_img = dst / r.final_split / "images" / out_name
        with Image.open(src) as im:
            if UNSTRETCH_SIZE and DATASETS[r.ds]["stretched_from_16_9"] and im.size[0] == im.size[1]:
                # deshace el "Stretch to 640x640": mismo aspecto 16:9 que el video, que YOLO letterboxea en inferencia
                im.convert("RGB").resize(UNSTRETCH_SIZE, Image.BICUBIC).save(out_img, quality=95)
            else:
                shutil.copy2(src, out_img)
        rows = read_labels(lbl)                                  # toda clase -> 0 (CLASS_NAMES)
        (dst / r.final_split / "labels" / f"{Path(out_name).stem}.txt").write_text(
            "\n".join(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for _, x, y, w, h in rows))
        manifest.append({"ds": r.ds, "split": r.final_split, "file": out_name, "sha1": hashlib.sha1(out_img.read_bytes()).hexdigest(),
                         "src": r.src, "frame": int(r.frame), "block": r.block})
    data_yaml = {"path": str(dst.resolve()), "train": "train/images", "val": "valid/images", "test": "test/images",
                 "names": names}
    (dst / "data.yaml").write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True))
    man = pd.DataFrame(manifest).sort_values(["split", "file"])
    man.to_csv(dst / "manifest.csv", index=False)
    digest = hashlib.sha1(man.to_csv(index=False).encode()).hexdigest()
    return dst / "data.yaml", digest

DATA_YAML, DATASET_DIGEST = materialize(imgs, DATA_FINAL, CLASS_NAMES)
# Ablación (ds2 solo) para medir si los frames de video de ds1 aportan en la prueba propia
DATA_YAML_DS2, DIGEST_DS2 = materialize(imgs[imgs.ds == "ds2"], WORK / "data" / "final_ds2", CLASS_NAMES)
RUN_RECORD.setdefault("splits", {}).update({"split_decision": SPLIT_DECISION, "final_yaml": str(DATA_YAML),
                              "manifest_sha1": DATASET_DIGEST, "manifest_sha1_ds2": DIGEST_DS2, "review_notes": REVIEW_NOTES})
print(DATA_YAML, DATASET_DIGEST)

### 4.1 Prueba de decisión: frames de los videos propios (CA-01.5.5)

247 frames (~6 fps) de 3 videos stock 1080p, etiquetados con pre-etiquetas YOLO26s y corregidos en Roboflow (`yilber-lopez/humming`): 174 cajas.
Se excluyen 4 frames ambiguos (`ignorar_frames.txt`). Estos frames no se usan para entrenar ni para elegir hiperparámetros.


In [ ]:
import zipfile
VIDEO_EVAL_DIR = Path("eval_frames")   # carpeta video-val/eval_frames (zips de frames + prueba_etiquetas_corregidas.zip)
TEST_VIDEO = DATA_FINAL / "test_video"

def build_video_test(src_dir: Path, dst: Path) -> Path:
    (dst / "images").mkdir(parents=True, exist_ok=True); (dst / "labels").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src_dir / "prueba_etiquetas_corregidas.zip") as z:
        ignore = set(z.read("ignorar_frames.txt").decode().split())
        labels = {Path(n).stem: z.read(n).decode() for n in z.namelist() if n.startswith("labels/") and n.endswith(".txt")}
    n = 0
    for fz in sorted(src_dir.glob("*_frames.zip")):
        with zipfile.ZipFile(fz) as z:
            for name in z.namelist():
                stem = Path(name).stem
                if name.startswith("images/") and stem not in ignore:
                    assert stem in labels, f"{stem} sin etiqueta corregida"
                    (dst / "images" / f"{stem}.jpg").write_bytes(z.read(name))
                    (dst / "labels" / f"{stem}.txt").write_text(labels[stem]); n += 1
    print(f"Prueba de video: {n} frames, {sum(len(l.split()) // 5 for k, l in labels.items() if k not in ignore)} cajas")
    return dst

if VIDEO_EVAL_DIR.exists():
    build_video_test(VIDEO_EVAL_DIR, TEST_VIDEO)
    RUN_RECORD["test_video"] = {"dir": str(TEST_VIDEO), "source": "yilber-lopez/humming (etiquetas corregidas)"}
else:
    print("Copia video-val/eval_frames junto al notebook (o ajusta VIDEO_EVAL_DIR).")


## 5. Evaluador común y línea base YOLO26 COCO `bird` (US-02.2, CA-02.2.1)

Para comparar en igualdad de condiciones, la línea base (clase COCO 14 → nuestra clase 0) y el modelo ajustado se miden con **el mismo evaluador** sobre la misma partición de prueba: AP@0.5, precisión y recall a un umbral fijo, y recall por tamaño de caja.

In [ ]:
def iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """IoU entre cajas xyxy: a (N,4), b (M,4)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2]); br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(2)
    area_a = (a[:, 2:] - a[:, :2]).prod(1); area_b = (b[:, 2:] - b[:, :2]).prod(1)
    return inter / (area_a[:, None] + area_b[None] - inter + 1e-9)

def size_bin(w_px, h_px):
    s = np.sqrt(w_px * h_px)
    return "pequeña(<32px)" if s < 32 else ("mediana(32-96px)" if s < 96 else "grande(>=96px)")

def evaluate(pred_fn, split_dir: Path, iou_thr=0.5, conf_thr=0.25):
    """pred_fn(path) -> (K,5) [x1,y1,x2,y2,conf] en píxeles. Devuelve métricas y tabla por imagen."""
    scores, tps, n_gt, per_img, gt_rows = [], [], 0, [], []
    for img in sorted(p for p in (split_dir / "images").iterdir() if p.suffix.lower() in IMG_EXTS):
        with Image.open(img) as im: W, H = im.size
        gt = np.array([[(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H]
                       for _, x, y, w, h in read_labels(split_dir / "labels" / f"{img.stem}.txt")]).reshape(-1, 4)
        pr = np.asarray(pred_fn(img), float).reshape(-1, 5)
        pr = pr[np.argsort(-pr[:, 4])]
        ious = iou_matrix(pr[:, :4], gt); used = np.zeros(len(gt), bool); tp = np.zeros(len(pr), bool)
        for i in range(len(pr)):
            if len(gt):
                j = int(np.argmax(np.where(used, -1, ious[i])))
                if not used[j] and ious[i, j] >= iou_thr:
                    used[j] = tp[i] = True
        # recall a conf_thr por caja GT (para análisis por tamaño)
        keep = pr[:, 4] >= conf_thr
        ious_k = iou_matrix(pr[keep, :4], gt)
        for j, g in enumerate(gt):
            gt_rows.append(dict(path=str(img), size=size_bin(g[2] - g[0], g[3] - g[1]),
                                hit=bool(ious_k.shape[0] and ious_k[:, j].max() >= iou_thr)))
        scores += pr[:, 4].tolist(); tps += tp.tolist(); n_gt += len(gt)
        per_img.append(dict(path=str(img), n_gt=len(gt), n_pred=int(keep.sum()), tp=int((tp & keep).sum())))
    scores, tps = np.array(scores), np.array(tps, bool)
    order = np.argsort(-scores); tp_c = np.cumsum(tps[order]); fp_c = np.cumsum(~tps[order])
    rec = tp_c / max(n_gt, 1); prec = tp_c / np.maximum(tp_c + fp_c, 1)
    # AP de 101 puntos estilo COCO: precisión máxima con recall >= r; 0 si r no se alcanza
    ap50 = float(np.mean([prec[rec >= r].max() if (rec >= r).any() else 0.0 for r in np.linspace(0, 1, 101)]))
    k = scores >= conf_thr
    P = float(tps[k].sum() / max(k.sum(), 1)); R = float(tps[k].sum() / max(n_gt, 1))
    gt_df, img_df = pd.DataFrame(gt_rows), pd.DataFrame(per_img)
    by_size = gt_df.groupby("size")["hit"].agg(["mean", "size"]).rename(columns={"mean": "recall", "size": "n"})
    return {"AP50": ap50, f"P@{conf_thr}": P, f"R@{conf_thr}": R, "n_gt": n_gt}, by_size, img_df

def yolo_pred_fn(model, classes=None, class_map=None, imgsz=640):
    def fn(path):
        r = model.predict(str(path), conf=0.001, imgsz=imgsz, classes=classes, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            return np.zeros((0, 5))
        return np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None]])
    return fn

In [ ]:
import ultralytics, torch
from ultralytics import YOLO
RUN_RECORD.update({"ultralytics": ultralytics.__version__, "torch": torch.__version__,
                   "cuda": torch.cuda.is_available()})

TEST_RF = DATA_FINAL / "test"                         # prueba provisional (Roboflow)
TEST_DIR = TEST_VIDEO if TEST_VIDEO.exists() else TEST_RF  # prueba de decisión: videos propios
BASE_WEIGHTS = "yolo26n.pt"
baseline = YOLO(BASE_WEIGHTS)
base_metrics, base_by_size, base_imgs = evaluate(yolo_pred_fn(baseline, classes=[COCO_BIRD_ID]), TEST_DIR)
print(f"Línea base COCO 'bird' en {TEST_DIR.name}:", base_metrics)
display(base_by_size)
RUN_RECORD["baseline"] = {"weights": BASE_WEIGHTS, **base_metrics}

## 6. Entrenamiento YOLO26 (US-02.2)

Entrena con ds2 + ds1 (`DATA_YAML`) y, si `RUN_ABLATION`, también solo con ds2 (`DATA_YAML_DS2`). Empieza con `yolo26n` para iterar rápido; prueba `yolo26s` e `imgsz=960` solo si la evaluación lo justifica (CA-02.2.6, CA-02.2.7).

In [ ]:
RUN_ABLATION = os.environ.get("RUN_ABLATION", "1") == "1"
def _num(v: str):
    return float(v) if "." in v else int(v)
# Uso de recursos de Ultralytics: batch=-1 (auto, ~60 % de VRAM) o batch=0.70 (fracción de VRAM);
# cache="disk" guarda las imágenes decodificadas (.npy). En Windows, cache="ram" se duplica en cada worker y agota la RAM
# (probado 2026-11-01 en la laptop: 8 workers -> OOM). amp=True usa precisión mixta.
TRAIN_BASE = dict(epochs=int(os.environ.get("EPOCHS", 40)), imgsz=640, batch=_num(os.environ.get("BATCH", "0.70")),
                  cache=os.environ.get("CACHE", "disk") or False, amp=True,
                  workers=int(os.environ.get("WORKERS", 3)), fraction=float(os.environ.get("FRACTION", 1.0)), patience=int(os.environ.get("PATIENCE", 15)), seed=SEED, deterministic=True,
                  project=str(RUNS.resolve()), exist_ok=True, plots=True)   # ruta absoluta: Ultralytics antepone runs/detect a las relativas
configs = {"yolo26n_ds2_ds1": DATA_YAML, **({"yolo26n_ds2": DATA_YAML_DS2} if RUN_ABLATION else {})}

def train_resumable(name: str, data_yaml: Path) -> Path:
    """Entrena o reanuda: si hay last.pt sin marca de fin, continúa (resume=True); si ya terminó, no repite."""
    wdir = RUNS.resolve() / name / "weights"; done = wdir.parent / "TRAINING_DONE"
    if done.exists():
        print(f"{name}: ya entrenado, se reutiliza {wdir / 'best.pt'}")
    elif (wdir / "last.pt").exists():
        print(f"{name}: reanudando desde {wdir / 'last.pt'}")
        m = YOLO(str(wdir / "last.pt")); m.train(resume=True); wdir = Path(m.trainer.save_dir) / "weights"
    else:
        m = YOLO("yolo26n.pt"); m.train(data=str(data_yaml), name=name, **TRAIN_BASE); wdir = Path(m.trainer.save_dir) / "weights"
    done = wdir.parent / "TRAINING_DONE"
    done.write_text(dt.datetime.now().isoformat(timespec="seconds"))
    return wdir / "best.pt"

TRAINED = {name: train_resumable(name, y) for name, y in configs.items()}
BEST = TRAINED["yolo26n_ds2_ds1"]
TRAIN_CFG = {**TRAIN_BASE, "data": str(DATA_YAML)}
RUN_RECORD["train"] = {"base": "yolo26n.pt", **TRAIN_BASE, "runs": {k: str(v) for k, v in TRAINED.items()}}
print(TRAINED)


## 7. Evaluación (US-02.3)

Se mide cada modelo en la **prueba de video propia** (decisión) y en la prueba de Roboflow (provisional, mismo dominio que el entrenamiento).


In [ ]:
rows, by_size = {"linea_base_coco_bird": base_metrics}, {"base": base_by_size}
for name, w in TRAINED.items():
    m = YOLO(str(w))
    rows[name], by_size[name], imgs_eval = evaluate(yolo_pred_fn(m, imgsz=TRAIN_CFG["imgsz"]), TEST_DIR)
    if name == "yolo26n_ds2_ds1":
        tuned, tuned_metrics, tuned_by_size, tuned_imgs = m, rows[name], by_size[name], imgs_eval
comparison = pd.DataFrame(rows).T
display(comparison)
display(pd.concat(by_size, axis=1))
val_metrics = tuned.val(data=str(DATA_YAML), split="test", imgsz=TRAIN_CFG["imgsz"], plots=True,
                        project=str(RUNS.resolve()), name="test_rf_eval")
RUN_RECORD["eval"] = {"test_dir": str(TEST_DIR), "por_modelo": {k: dict(v) for k, v in rows.items()},
                      "rf_test_map50": float(val_metrics.box.map50), "rf_test_map50_95": float(val_metrics.box.map)}


In [ ]:
# Ejemplos de errores representativos (CA-02.2.5, CA-02.3.3)
def tuned_lookup(p):
    r = tuned.predict(str(p), conf=0.25, verbose=False)[0]
    return [(0, *b) for b in r.boxes.xywhn.cpu().numpy()] if r.boxes is not None else []

fn_imgs = tuned_imgs[tuned_imgs.tp < tuned_imgs.n_gt]["path"].tolist()[:12]
fp_imgs = tuned_imgs[tuned_imgs.n_pred > tuned_imgs.tp]["path"].tolist()[:12]
if fn_imgs: show_grid(fn_imgs, tuned_lookup, title="Falsos negativos (predicciones en verde)", out=REPORTS / "errores_fn.png")
if fp_imgs: show_grid(fp_imgs, tuned_lookup, title="Falsos positivos (predicciones en verde)", out=REPORTS / "errores_fp.png")

## 8. Tracking y trayectoria detallada (US-02.1, US-02.4, US-02.5), **requiere video**

- `persist=True` sobre frames consecutivos; el tracker se indica siempre (Ultralytics usa TrackTrack si no se indica).
- `conf` bajo en la llamada (0.1) para que ByteTrack pueda usar detecciones de baja confianza en su segunda asociación.
- Marca temporal: `CAP_PROP_POS_MSEC` del decodificador, además del índice de frame (CA-01.2.3).
- La salida CSV es la **fuente de verdad** y no se sobrescribe; las pérdidas quedan como frames sin fila para ese ID.

In [ ]:
import cv2
from ultralytics.utils import ROOT as ULTRA_ROOT

VIDEO_PATH = None          # p. ej. Path("videos/colibri_01.mp4") cuando el video esté disponible
TRACKERS = {"bytetrack": "bytetrack.yaml", "ocsort": "ocsort.yaml"}
# Ajustar track_buffer a la duración de las pérdidas observadas (CA-02.4.1).
# OC-SORT trae use_byte=False por defecto; se activa para comparar en igualdad con ByteTrack.
TRACKER_OVERRIDES = {"bytetrack": {"track_buffer": 60}, "ocsort": {"track_buffer": 60, "use_byte": True}}

def tracker_cfg(name: str) -> Path:
    """Copia versionada del YAML del tracker con los ajustes del proyecto."""
    src = ULTRA_ROOT / "cfg" / "trackers" / TRACKERS[name]
    cfg = yaml.safe_load(src.read_text())
    cfg.update({k: v for k, v in TRACKER_OVERRIDES.get(name, {}).items() if k in cfg})
    if "gmc_method" in cfg:
        cfg["gmc_method"] = "none"        # cámara fija
    dst = REPORTS / f"tracker_{name}.yaml"
    dst.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return dst

def track_video(weights: Path, video: Path, tracker: str, conf=0.1, imgsz=640) -> pd.DataFrame:
    model = YOLO(str(weights))                  # modelo nuevo por tracker: estado limpio
    cfg_path = tracker_cfg(tracker)
    cap = cv2.VideoCapture(str(video)); fps = cap.get(cv2.CAP_PROP_FPS)
    rows, frame_idx = [], 0
    while True:
        t_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        ok, frame = cap.read()
        if not ok:
            break
        r = model.track(frame, persist=True, tracker=str(cfg_path), conf=conf, imgsz=imgsz, verbose=False)[0]
        if r.boxes is not None and r.boxes.is_track:
            for (cx, cy, w, h), tid, c in zip(r.boxes.xywh.cpu().numpy(), r.boxes.id.int().cpu().tolist(),
                                             r.boxes.conf.cpu().numpy()):
                rows.append(dict(frame=frame_idx, t_ms=t_ms, track_id=tid, cx_px=cx, cy_px=cy,
                                 w_px=w, h_px=h, conf=float(c)))
        frame_idx += 1
    cap.release()
    df = pd.DataFrame(rows)
    df.attrs.update({"fps_declared": fps, "n_frames": frame_idx, "tracker": tracker, "video": str(video)})
    return df

def track_summary(df: pd.DataFrame) -> dict:
    n_frames = df.attrs["n_frames"]
    lens = df.groupby("track_id").size()
    covered = df["frame"].nunique()
    gaps = df.groupby("track_id")["frame"].apply(lambda f: int((np.diff(np.sort(f.values)) > 1).sum()))
    return {"tracker": df.attrs["tracker"], "ids": int(lens.size), "frames_con_deteccion": covered,
            "cobertura": round(covered / max(n_frames, 1), 3), "huecos_internos": int(gaps.sum()),
            "longitud_mediana_track": float(lens.median()) if len(lens) else 0}

if VIDEO_PATH and Path(VIDEO_PATH).exists():
    out_dir = WORK / "trayectorias"; out_dir.mkdir(exist_ok=True)
    summaries = []
    for name in TRACKERS:
        df = track_video(BEST, Path(VIDEO_PATH), name)
        out = out_dir / f"{Path(VIDEO_PATH).stem}__{name}__detallada.csv"
        assert not out.exists(), f"{out} ya existe: la trayectoria detallada no se sobrescribe"
        df.to_csv(out, index=False)
        (out.with_suffix(".meta.json")).write_text(json.dumps(df.attrs, indent=2))
        summaries.append(track_summary(df))
    display(pd.DataFrame(summaries))
else:
    print("Sin video todavía: sección 8 pendiente (US-01.1 bloqueada).")

In [ ]:
def overlay_trajectory(video: Path, df: pd.DataFrame, out: Path, tail=60):
    """Video de control con la trayectoria superpuesta (CA-02.5.5)."""
    cap = cv2.VideoCapture(str(video)); fps = cap.get(cv2.CAP_PROP_FPS)
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    vw = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    by_frame = df.groupby("frame"); idx = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        hist = df[(df.frame <= idx) & (df.frame > idx - tail)]
        for tid, g in hist.groupby("track_id"):
            pts = g.sort_values("frame")[["cx_px", "cy_px"]].to_numpy(int)
            color = tuple(int(c) for c in np.random.default_rng(tid).integers(60, 255, 3))
            cv2.polylines(frame, [pts.reshape(-1, 1, 2)], False, color, 2)
            cv2.putText(frame, f"id {tid}", tuple(pts[-1] + 5), cv2.FONT_HERSHEY_SIMPLEX, .6, color, 2)
        vw.write(frame); idx += 1
    cap.release(); vw.release()

## 9. Registro de la ejecución (DoD-03)

In [ ]:
(REPORTS / "run_record.json").write_text(json.dumps(RUN_RECORD, indent=2, default=str, ensure_ascii=False))
print(json.dumps(RUN_RECORD, indent=2, default=str, ensure_ascii=False))